# Bible ChatBot - Telugu Version


In [1]:
# load environment variables
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
astra_db_endpoint = os.getenv("ASTRADB_ENDPOINT")
astra_db_token = os.getenv("ASTRADB_APPLICATION_TOKEN")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

## Loading Data


In [2]:
from langchain_community.document_loaders.pdf import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("../data/Telugu")

docs = loader.load()

print(f"Loaded {len(docs)} documents")

Loaded 1517 documents


In [3]:
docs[0]

Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'pandoc', 'creationdate': '', 'title': 'ఆదికాండము', 'source': '..\\data\\Telugu\\1.pdf', 'total_pages': 70, 'page': 0, 'page_label': '1'}, page_content='ఆదికాండము\n1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19\n20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38\n39 40 41 42 43 44 45 46 47 48 49 50\nచా ప్ట ర్ 1\nఆదియందు దేవుడు భూమ్యాకాశములను సృయంచెను.\n2 భూమి నిరాకారముగాను శూమ్యాముగాను యండెను; చీకటి అగాధ జలము పై  కమిమ్మియండెను; దేవుని ఆమ్మి జలముల పై \nఅ ల్లా డుయండెను.\n3 దేవుడు వెలుగు కమ్మిని పలుకగా వెలుగు కలిగెను.\n4 వెలుగు యంచి పై ట్టు  దేవుడుచూచెను; దేవుడు వెలుగును చీకటిని వేరుపరచెను.\n5 దేవుడు వెలుగుకు పగలని, చీకటికి రా త్రి  అని పేరు ట్టు ను. అ స్త మును దమును కలుగగా ఒక\nదియెను.\n6 రి దేవుడుజలముల ధమ్యా నొక విశాలము కలిగి ఆ జలములను ఈ జలములను వేరుపరను గాకని పలికెను.\n7 దేవుడు ఆ విశాలము చేసి విశాలము కి త్రి యంది జలములను విశాలము మీది జలములను వేరుపరపగా ఆ ప త్రి కారయెను.\n8 దేవుడు ఆ విశాలముకు ఆకాశని పేరు ట్టు ను. అ స్త ము

### Splitting the data into chunks


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

print(f"Splittting {len(docs)} to {len(splits)} chunks")

Splittting 1517 to 2955 chunks


### Creating an Embedding model


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "l3cube-pune/telugu-sentence-bert-nli"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
embeddings

pytorch_model.bin:   0%|          | 0.00/950M [00:00<?, ?B/s]

d:\Documents\bible_chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amuly\.cache\huggingface\hub\models--l3cube-pune--telugu-sentence-bert-nli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/518 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/950M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFaceEmbeddings(model_name='l3cube-pune/telugu-sentence-bert-nli', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## Creating a vector store and making embeddings


In [ ]:
from langchain_astradb import AstraDBVectorStore



vector_store = AstraDBVectorStore(collection_name="telugu_bible",
                                  embedding=embeddings,
                                  token=astra_db_token,
                                  api_endpoint=astra_db_endpoint,
                                  )

In [7]:
# Encode documents for duplicate prevention
import hashlib

from langchain_core.documents import Document


def generate_doc_id(doc: Document):
    return hashlib.sha256(doc.page_content.encode()).hexdigest()

In [ ]:
# ## RUN only once
# # upload documents by preventing duplicates
# import time

# start = time.time()
# ids = [generate_doc_id(doc) for doc in splits]

# vector_store.add_documents(splits, ids=ids)

# print(f'embedded and uploaded {len(splits)} in {time.time() - start} seconds')

embedded and uploaded 2955 in 764.7522041797638 seconds


### Querying the vector store


In [ ]:
sim_docs = vector_store.similarity_search("దుష్టుల ఆలోచన చొప్పున నడువక")
sim_docs

### Creating prompt template 

In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "You are a helpful assistant that responds **only in Telugu**.\n"
            "Use the provided context to answer the user's question.\n"
            "Do not make up or hallucinate any information.\n\n"
            "Context:\n{context}"
        ),
    ),
    ("human", "{input}"),
])



In [11]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_groq import ChatGroq

llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile")
docs_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)
retriever = vector_store.as_retriever()

chain = create_retrieval_chain(retriever, docs_chain)

In [12]:
query = "keerthanlu 1:1 - display the context?"
response = chain.invoke({"input": query})

In [13]:
response['answer']

'కీర్తనలు 1:1\n\nదుష్టుల ఆలోచనచొప్పున నడువక, పాపుల మార్గమున నిలువక, అపహాసకులు కూర్చుండు చోటను కూర్చుండక.'

In [14]:
response['context']

[Document(id='b282b02aba60aedaa462a2abeb8e83042357ff728f13db57d5e3f1c0a19674d9', metadata={'producer': 'WeasyPrint 62.3', 'creator': 'pandoc', 'creationdate': '', 'title': 'కీర్తనల గ్రంథము', 'source': '..\\data\\Telugu\\19.pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}, page_content='కీ ర్త నల గ్రంథము\n1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19\n20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38\n39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57\n58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76\n77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95\n96 97 98 99 100101102103104105106107108109110111112113114\n115116117118119120121122123124125126127128129130131132133\n134135136137138139140141142143144145146147148149150\nచా ప్ట ర్ 1\nదు ష్టు ల ఆలోచనచొప్పున నడువకపాపుల మా ర్గ మున నిలువక అపహాసకులు కూర్చుండు చోటను కూర్చుండక\n2 యెహోవా ధర్మశాస స్త్ర మునర్చుందు ఆనర్చుంర్చుంచుచువారా త్ర ము దానిని ధ్యానిర్చుంచువాడు ధనుధ్యాడు.\n3 అడు నీటికాలువల యోను నాటబడిన దై ఆక